# Instructions: How to Use Python Scripting to Fetch Data from OpenAlex

This script retrieves publication metadata from the [OpenAlex API](https://openalex.org/) for a specific institution (default: Utah State University) and saves the results to a `.csv` file.

Follow the steps below to customize and run the script for your needs.

---

## 1. Set the Institution

```python
institution_id = "https://openalex.org/i121980950"
```
- This ID corresponds to Utah State University. 
- To fetch data for another institution, replace it with the appropriate OpenAlex institution ID.
    - Format: `"https://openalex.org/iXXXXXXXXX"`

## 2. Set the Output File Name
```python
output_file = "Title.csv"
```
- This is the name of the output file the script will generate.
- Change it to something descriptive, e.g., `"OpenAlex_USU_2025.csv"`.
- Make sure the filename:
    - Is inside quotation marks `" "`
    - Ends with `.csv`

## 3. Adjust the API Request Parameters
```python
params = {
    "filter": f"institution.id:{institution_id},from_publication_date:2019-01-01,to_publication_date:2025-07-15",
    "per_page": "200",
    "mailto": "your.email@usu.edu",
    "cursor": "*"
}
```
- `filter`: Specifies the date range and institution.
- `per_page`: Number of records returned per request (maximum is 200).
- `mailto`: Replace with your USU email address. OpenAlex uses this to monitor usage.
- `cursor`: Used for pagination — leave this as "*" to start at the beginning.
- You are allowed 100,000 API requests per month per email address.

## 4. Customize Column Headers in the Output File
```python
writer.writerow([
    "Title", "Journal", "Publisher", "Date", "in_scopus", "type", "is_oa",
    "version", "is_accepted", "is_published", "license", "DOI"
] + author_headers)
```
- This list defines the column names that appear in your .csv file.
- You can:
    - Add more headers to collect additional fields
    - Remove ones you don’t need
- Ensure that the headers match the order of data in the `writer.writerow()` line later in the script.

## 5. Specify Metadata to Extract
```python
title = work.get('title', '')
date = work.get('publication_date', '')
doi = work.get('doi', '')
is_oa = work.get('open_access', {}).get('is_oa', '')
...
```
- These lines extract the values for each work (publication).
- You can add additional metadata fields (issn, OA status, etc.).
- To explore what's available, view a sample OpenAlex Work object in JSON format at: https://api.openalex.org/works/W3103145119
    - Click pretty-print to make the JSON object easier to read.

## 6. Assign Extracted Data to the CSV
```python 
writer.writerow([
    title, journal, publisher, date, in_scopus, work_type,
    is_oa, version, is_accepted, is_published, license, doi
] + authors_data)
```
- This writes the extracted data to the .csv file.
- If you added new metadata fields or removed any, update this list to match the headers.

## 7. Author Data Structure
- The script captures data for up to 200 authors per publication.
- For each author, it records:
    - `Author{i}Name`
    - `Author{i}Affiliation`
    - `Author{i}ORCid`
- If there are fewer than 200 authors, the remaining fields are left blank (for consistent column structure).

## 8. Run the Script
- Run the script in the cell below to retreive the data.
- While running, the script will:
    - Print the current page (`cursor`)
    - Fetch data page-by-page until complete
    - Save results into your `.csv` file

## Final Notes and Important Considerations
- If you need more assistance with step 5, see the last cell to explore a JSON object from OpenAlex in this notebook.
- If you are requesting a large volume of data, I recommend limiting the number of records you retrieve per call until it looks and functions the way you intend.
    - You can do this by removing the `#` at the beginning of these lines in the code to enable the limit:
        ```python 
        # count = 0
        # max_entries = 200 
        ```
    and:

    ```python
                # count += 1
                    # if count >= max_entries:
                        # break
    
            # if count >= max_entries:
                # break
    ```
    - Make sure to add the `#` back in when you're happy with the results to retrieve all the data.


In [ ]:
import requests
import csv
import time

institution_id = "https://openalex.org/i121980950"
output_file = "Title.csv"

base_url = "https://api.openalex.org/works"

params = {
    "filter": f"institution.id:{institution_id},from_publication_date:2019-01-01,to_publication_date:2025-07-15",
    "per_page": "200",
    "mailto": "your.email@usu.edu",
    "cursor": "*"
}

# count = 0
# max_entries = 200 

author_headers = []
for i in range(1, 201):
    author_headers.extend([
        f"Author{i}Name", f"Author{i}Affiliation", f"Author{i}ORCid"
    ])

with open(output_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow([
        "Title", "Journal", "Publisher", "Date", "in_scopus", "type", "is_oa",
        "version", "is_accepted", "is_published", "license", "DOI"
    ] + author_headers)

    while True:
        print(f"Fetching: {params['cursor']}")
        response = requests.get(base_url, params=params)

        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            break

        data = response.json()

        for work in data.get('results', []):
            title = work.get('title', '')
            date = work.get('publication_date', '')
            work_type = work.get('type_crossref', '')
            doi = work.get('doi', '')
            is_oa = work.get('open_access', {}).get('is_oa', '')

            journal = ''
            publisher = ''
            in_scopus = ''
            version = ''
            is_accepted = ''
            is_published = ''

            primary_location = work.get('primary_location')
            if primary_location:
                version = primary_location.get('version', '')
                is_accepted = primary_location.get('is_accepted', '')
                is_published = primary_location.get('is_published', '')
                license = primary_location.get('license', '')
                source = primary_location.get('source')
                if isinstance(source, dict):
                    journal = source.get('display_name', '')
                    in_scopus = source.get('is_indexed_in_scopus', '')
                    publisher_lineage = source.get('host_organization_lineage_names', [])
                    publisher = publisher_lineage[-1] if publisher_lineage else ''
            else:
                license = ''

            authors_data = []
            authorships = work.get('authorships', [])
            for auth in authorships[:200]:
                name = auth.get('author', {}).get('display_name', '')
                orcid = auth.get('author', {}).get('orcid', '')
                affiliations = auth.get('raw_affiliation_strings', [])
                affil = ", ".join(affiliations) if affiliations else ''
                authors_data.extend([name, affil, orcid])

            while len(authors_data) < 600: 
                authors_data.extend(['', '', ''])

            writer.writerow([
                title, journal, publisher, date, in_scopus, work_type,
                is_oa, version, is_accepted, is_published, license, doi
            ] + authors_data)

            # count += 1
            # if count >= max_entries:
                # break
    
        # if count >= max_entries:
            # break

        next_cursor = data.get('meta', {}).get('next_cursor')
        if next_cursor:
            params['cursor'] = next_cursor
            time.sleep(1)
        else:
            break

print(f"Data saved to {output_file}")


## Exploring JSON Objects

### Understanding the Data Extraction Block
The section of the script that starts with: 
```python
for work in data.get('results', []):
```
...is where the script loops through each publiation record returned by OpenAlex and extracts the metadata we want to save.

### How to Read OpenAlex JSON
OpenAlex returns structured data in JSON format, which is basically a nested dictionary. Here's a simple example structure for one publication:
```json
{
  "title": "Your Paper Title",
  "publication_date": "2022-01-01",
  "doi": "https://doi.org/...",
  "open_access": {
    "is_oa": true
  },
  "primary_location": {
    "version": "publishedVersion",
    "is_accepted": true,
    "is_published": true,
    "license": "cc-by",
    "source": {
      "display_name": "Nature Methods",
      "is_indexed_in_scopus": true,
      "host_organization_lineage_names": ["Nature Portfolio", "Springer Nature"]
    }
  },
  "authorships": [
    {
      "author": {
        "display_name": "Jane Doe",
        "orcid": "0000-0002-1234-5678"
      },
      "raw_affiliation_strings": ["Utah State University"]
    }
  ]
}
```
You can also view this structure with a real publication by going to OpenAlex: https://api.openalex.org/works/W3103145119

### What Each Line in the Loop Does
1. Basic Metadata
```python 
title = work.get('title', '')
date = work.get('publication_date', '')
work_type = work.get('type_crossref', '')
doi = work.get('doi', '')
is_oa = work.get('open_access', {}).get('is_oa', '')
```
- Extracts the title, date, publication type, DOI, and open access status.
- `work.get(..., '')` means: get this field or return an empty string if it's missing.
- `open_access` is nested, so we use `.get('open_access', {})` to avoid errors if that field is missing.
    - You can see this yourself be running the cell below, and clicking on open_access in the output. You should see a list of metadata values: `is_oa`, `oa_status`, `oa_url` and `any_repository_has_fulltext`. Each of these values are "nested" inside `open_access`.
    - If you wanted to retreive `oa_status` instead os `is_oa`, you would change this in the script:
    ```python
    is_oa = work.get('open_access', {}).get('is_oa', '')  
    ```
    to:
    ```python
    oa_status = work.get('open_access', {}).get('oa_status', '')  
    ```
    If you want to retrive both `is_oa` and `oa_status`, you would put both:
    ```python
    is_oa = work.get('open_access', {}).get('is_oa', '')
    oa_status = work.get('open_access', {}).get('oa_status', '')
    ```
    ... and then update the columns in the `writer.writerow([...])` sections of the code.


2. Default Empty Fields (in case data is missing)
```python 
journal = ''
publisher = ''
in_scopus = ''
version = ''
is_accepted = ''
is_published = ''
```
- Sets default values in case a field is not found or available in the JSON response.

3. `primary_location` Metadata
```python 
primary_location = work.get('primary_location')
if primary_location:
    version = primary_location.get('version', '')
    is_accepted = primary_location.get('is_accepted', '')
    is_published = primary_location.get('is_published', '')
    license = primary_location.get('license', '')
    source = primary_location.get('source')
```
- `primary_location` describes the main place the article was published.
- Nested inside it we look for:
    - `version`: accepted/published manuscript
    - `is_accepted` / `is_published`: status flags
    - `license`: e.g., "cc-by"
- You can see this yourself by clicking on `primary_location`, and seeing the various metadata fields stored there.+
- Again, if you wanted to fetch the the `pdf_url`, you could either update one of the lines in the code:
    ```python
    is_accepted = primary_location.get('is_accepted', '')
    ```
    to:
    ```python
    pdf_url = primary_location.get('pdf_url', '')
    ```
    or, you could add `pdf_url` after `license`:
    ```python
    license = primary_location.get('license', '')
    pdf_url = primary_location.get('pdf_url', '')
    source = primary_location.get('source')
    ```
    However, it **cannot** come after `source`!

4. Journal Info from `source`
```python
if isinstance(source, dict):
    journal = source.get('display_name', '')
    in_scopus = source.get('is_indexed_in_scopus', '')
    publisher_lineage = source.get('host_organization_lineage_names', [])
    publisher = publisher_lineage[-1] if publisher_lineage else ''
```
- `source` is the journal or hosting platform.
- We extract:
    - `display_name`: Journal title
    - `is_indexed_in_scopus`: True/False
    - `publisher`: Last item in `host_organization_lineage_names` (the top-level publisher, like Springer or Elsevier)
        - If you want to obtain the imprints (i.e. Nature Portfolio, Cell Press, etc.), you would specify:
            ```python
            publisher = publisher_lineage[0] if publisher_lineage else ''
            ```
            Alternatively, if you wanted to gather all publishers, you might specify:
            ```python
            publisher1 = publisher_lineage[0] if len(publisher_lineage) > 0 else ''
            publisher2 = publisher_lineage[1] if len(publisher_lineage) > 1 else ''
            publisher3 = publisher_lineage[2] if len(publisher_lineage) > 2 else ''
            ```
            ... and update the columns in the `writer.writerow([...])` sections of the code by adding in `publisher1`, `publisher2`, and `publisher3`.
            If you only want the first and last publisher in the lineage, you might specify:
            ```python
            publisher_first = publisher_lineage[0] if len(publisher_lineage) > 0 else ''
            publisher_last = publisher_lineage[-1] if len(publisher_lineage) > 1 else publisher_first
            ```
            ... and update the columns in the `writer.writerow([...])` sections of the code accordingly.
- Like other sections you can update or add metadatafeilds by specifying them before `publisher_lineage`, and adding the name of that field to the column headers in the `writer.writerow([...])` sections. 

5. Authors and Affiliations
```python
authors_data = []
authorships = work.get('authorships', [])
for auth in authorships[:200]:
    name = auth.get('author', {}).get('display_name', '')
    orcid = auth.get('author', {}).get('orcid', '')
    affiliations = auth.get('raw_affiliation_strings', [])
    affil = ", ".join(affiliations) if affiliations else ''
    authors_data.extend([name, affil, orcid])
```
- Loops through up to 200 authors:
  - Extracts their name, ORCID, and affiliations.
  - authorships is a list of dictionaries, one for each author.
  - extend([name, affil, orcid]) appends this author’s data to the final row.`
  - This limits the number of authors for a given work to 200. If you need to account for more authors for a single work, you may change `for auth in authorships[:200]` to `for auth in authorships[:x]`, and then you will need to update `while len(authors_data) < 600` to `while len(authors_data) < x*3`

Final Thoughts and Considerations
- This is not an exhaustive list of OpenAlex metadata fields — for more, check the OpenAlex Technical Docs at https://docs.openalex.org/
- The code in this notebook is not intended to be production-grade but is designed to be functional and customizable.
- Be careful with nested fields and make sure to update both your headers and row-writing logic if you add more metadata fields.
            
        


In [1]:
import requests
import json
from IPython.display import display, HTML

def display_json_collapsible(data, level=0):
    html = ""
    if isinstance(data, dict):
        for key, value in data.items():
            if isinstance(value, (dict, list)):
                html += f"<details style='margin-left:{level*15}px'><summary><b>{key}</b></summary>{display_json_collapsible(value, level+1)}</details>"
            else:
                html += f"<div style='margin-left:{level*15}px'><b>{key}:</b> {value}</div>"
    elif isinstance(data, list):
        for i, item in enumerate(data):
            html += f"<details style='margin-left:{level*15}px'><summary>Item {i}</summary>{display_json_collapsible(item, level+1)}</details>"
    else:
        html += str(data)
    return html

url = "https://api.openalex.org/works/W3103145119"
response = requests.get(url)
data = response.json()

display(HTML(display_json_collapsible(data)))
